# 실습 — RAG로 사내규정 챗봇 만들기

**현대경제연구소 사내 헬프데스크 챗봇 '현경봇' 만들기**

> ⚠️ 이 노트북의 규정은 **가상 규정**입니다.

## 시나리오

현대경제연구소에 입사한 신입 연구원 **김연구** 씨는 궁금한 게 많습니다.
"재택근무 돼요?", "부산 출장 숙박비는 얼마까지?", "ChatGPT 써도 되나요?"

인사팀·재무팀·IT팀에 매번 물어보기 번거로우니, **사내규정을 보고 답하는 챗봇**을 만들어 봅니다.

| 단계 | 내용 |
|------|------|
| 1️⃣ 준비 | 도구 설치, API 키, AI 불러보기 |
| 2️⃣ 문제 확인 | AI는 우리 회사 규정을 모른다 (환각) |
| 3️⃣ 역할 주기 | 프롬프트로 '현경봇' 역할 주기 |
| 4️⃣ RAG | 규정 문서를 같이 줘서 정확하게 답하기 |
| 5️⃣ 미션 | 나만의 질문으로 챗봇 테스트 |

## 📝 실습 방법

- 코드 속 `____` (밑줄 4개)를 알맞은 코드로 바꾼 뒤 실행하세요.
- 💡 힌트를 참고하세요. 막히면 **위니브 카페 노트북([Day5_1])** 의 같은 셀을 보면 됩니다.

# 1️⃣ 단계 — 준비

### 셀 1 — 도구 설치

In [ ]:
!pip install -q langchain langchain-openai python-dotenv

### 셀 2 — API 키 넣기

같은 폴더의 `.env` 파일에 `OPENAI_API_KEY=sk-...` 한 줄을 적어 두세요.

> 💡 LangSmith 키가 없으면 `401 Unauthorized` 경고가 계속 뜹니다. 아래 셀에서 추적을 꺼서 막아 둡니다.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()



if os.environ.get("OPENAI_API_KEY"):
    print("✅ 준비 완료! 다음 셀로 넘어가세요.")
else:
    print("❌ OPENAI_API_KEY가 없습니다. .env 파일을 확인하세요.")

### 셀 3 — AI 불러보기

`gpt-4o-mini` 모델을 `temperature=0`(안정적인 답)으로 준비합니다.

In [ ]:
from langchain_openai import ChatOpenAI

# TODO: 모델 이름 "gpt-4o-mini", temperature는 0
llm = ChatOpenAI(model=____, temperature=____)

# TODO: llm에게 질문하는 메서드 이름은?
answer = llm.____("경제연구소는 어떤 일을 하는 곳인지 한 문장으로 알려줘.")

print(answer.content)

# 2️⃣ 단계 — AI는 우리 회사 규정을 모른다

### 셀 4 — 사내규정 질문하기

규정에는 **"미사용 연차는 최대 5일 이월, 다음 해 3월 31일까지 사용"** 이라고 적혀 있습니다.
AI는 이 규정을 본 적이 없습니다. 어떻게 답할까요?

In [ ]:
question = "현대경제연구소에서 못 쓴 연차는 다음 해로 며칠까지 넘길 수 있나요?"

# TODO: llm에게 question을 묻고, 답의 글자만 출력하세요
answer = ____
print(answer.____)

> ⚠️ AI가 "회사마다 다르다", "보통 ○일" 같은 일반론이나 **지어낸 숫자**로 답했을 겁니다.
> 정답(최대 5일, 3월 31일까지)과 비교해 보세요. 이게 **환각**입니다.

> ✍️ AI의 답에서 틀린 부분을 적어보세요: 

# 3️⃣ 단계 — '현경봇'에게 역할 주기

### 셀 5 — 프롬프트로 사내 헬프데스크 역할 주기

- `system`: 챗봇의 역할·말투
- `human`: 직원 질문이 들어갈 자리 `{question}`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    # TODO: 현경봇의 역할을 한국어로 적어보세요 (예: 현대경제연구소 사내 헬프데스크 챗봇, 존댓말)
    
])

# TODO: 프롬프트 → AI → 글자만 추출, 세 가지를 | 로 연결
chatbot = ____ | ____ | ____

print(chatbot.invoke({"question": "신입인데 재택근무 할 수 있나요?"}))

> 💡 말투는 '현경봇'다워졌지만, **내용은 여전히 추측**입니다.
> (규정: 입사 3개월 미만은 재택 불가, 그 이후 주 2회까지)

# 4️⃣ 단계 — 규정 문서를 같이 줘서 정확하게 (RAG)

### 셀 6 — 사내규정 문서 준비

규정 3개를 변수에 담습니다.

| 변수 | 내용 | 담당 |
|------|------|------|
| `leave_policy` | 근무·휴가 (재택, 연차, 경조휴가) | 인사팀 |
| `travel_policy` | 출장·경비 (숙박비, 법인카드) | 재무팀 |
| `security_policy` | 정보보안 (보안등급, 외부 AI 사용) | IT팀 |

In [ ]:
# ※ 교육용 가상 규정입니다. 실제 기관의 규정과 무관합니다.

leave_policy = """# 현대경제연구소 근무·휴가 규정

## 1. 근무시간
- 기본 근무시간은 09:00~18:00 이며, 점심시간은 12:00~13:00 입니다.
- **시차출퇴근제**: 08:00~10:00 사이에 출근 시각을 선택할 수 있으며, 하루 8시간을 근무합니다.

## 2. 재택근무
- 재택근무는 **주 2회까지** 가능합니다.
- 전날 17시까지 그룹웨어에서 신청하고 팀장 승인을 받아야 합니다.
- **입사 3개월 미만 직원은 재택근무를 할 수 없습니다.**

## 3. 연차휴가
| 근속 기간 | 연차 일수 |
|-----------|-----------|
| 1년 미만 | 1개월 개근 시 1일 (최대 11일) |
| 1년 이상 | 15일 |
| 3년 이상 | 2년마다 1일 가산 (최대 25일) |

- 반차(0.5일)는 오전(09:00~13:00), 오후(14:00~18:00) 단위로 사용합니다. **반반차는 없습니다.**
- 연차는 사용 **3영업일 전**까지 그룹웨어로 신청합니다.
- **3일 이상 연속** 사용 시 **2주 전**까지 신청해야 합니다.
- 긴급한 경우 당일 09시 전 팀장에게 연락하고, 복귀 후 사후 신청합니다.

## 4. 미사용 연차
- 미사용 연차는 다음 해로 **최대 5일까지 이월**할 수 있습니다.
- 이월된 연차는 **다음 해 3월 31일까지** 사용해야 하며, 기한이 지나면 소멸합니다.
- 연차사용촉진제를 시행하므로, 이월분을 제외한 미사용 연차는 금전 보상하지 않습니다.

## 5. 경조휴가 (유급)
| 구분 | 휴가 | 경조금 |
|------|------|--------|
| 본인 결혼 | 5일 | 50만원 |
| 자녀 결혼 | 1일 | 30만원 |
| 배우자 출산 | 10일 | 20만원 |
| 부모·배우자 부모 사망 | 5일 | 50만원 |
| 조부모 사망 | 3일 | 20만원 |

- 배우자 출산휴가는 출산일로부터 90일 이내에 사용해야 하며, 1회 분할 사용이 가능합니다.

## 6. 병가
- 유급 병가는 연 5일입니다.
- 3일 이상 연속 병가 시 진단서를 제출해야 합니다.
"""

travel_policy = """# 현대경제연구소 출장·경비 규정

## 1. 출장 신청
- 국내 출장: 출발 **3영업일 전**까지 그룹웨어 품의, 팀장 승인
- 해외 출장: 출발 **2주 전**까지 품의, 본부장 승인

## 2. 국내 출장비
| 항목 | 기준 |
|------|------|
| 일비 | 1일 20,000원 |
| 식비 | 실비, 1끼 12,000원 한도 |
| 숙박비(서울) | 1박 120,000원 한도 |
| 숙박비(광역시) | 1박 100,000원 한도 |
| 숙박비(기타 지역) | 1박 80,000원 한도 |

- 교통: KTX·SRT 일반실 기준. 특실 이용 시 차액은 본인 부담입니다.
- 자가용 이용 시 km당 250원과 통행료를 지급합니다.
- 택시는 대중교통이 없는 구간이거나 22시 이후에만 인정됩니다.

## 3. 해외 출장비
- 항공권: 이코노미석 (임원은 비행 8시간 이상 시 비즈니스석)
- 일비: 1일 USD 50
- 숙박비: 북미·유럽 1박 USD 200, 그 외 지역 1박 USD 150 한도

## 4. 경비 정산
- 출장 복귀 후 **7영업일 이내**에 영수증을 첨부해 정산합니다.
- 기한을 넘기면 재무팀 승인 후에만 정산이 가능합니다.

## 5. 법인카드 사용 제한
다음은 법인카드로 결제할 수 없습니다.
- 개인 용품, 상품권, 유흥업소
- 23시 이후 결제 (출장 중 불가피한 교통비 제외)
- 주류 단독 결제 (공식 회식 제외)

## 6. 회식·접대비
- 팀 회식비: 분기별 1인당 30,000원
- 외부 접대비: 1인당 50,000원 한도, 초과 시 사전 본부장 승인 필요
"""

security_policy = """# 현대경제연구소 정보보안 규정

## 1. 문서 보안등급
| 등급 | 설명 | 외부 반출 |
|------|------|-----------|
| 대외비 | 미발표 연구결과, 경제전망 수치, 고객사 자료 | 금지 |
| 사내한 | 내부 회의자료, 업무 매뉴얼 | 팀장 승인 시 가능 |
| 공개 | 발간된 보고서, 보도자료 | 가능 |

- 대외비 문서 출력 시 워터마크가 자동으로 찍힙니다.

## 2. 엠바고(보도 유예)
- 경제전망 수치는 **공식 발표 전 외부에 언급할 수 없습니다.**
- 언론 문의는 개인이 답하지 말고 반드시 **홍보팀**으로 연결합니다.

## 3. 외부 생성형 AI 사용
- ChatGPT 등 외부 AI에는 **공개 등급 자료만** 입력할 수 있습니다.
- 대외비 자료, 미발표 전망치, 고객사 정보, 개인정보 입력은 **금지**입니다.
- 대외비 자료는 사내 AI 서비스 **'현경 어시스턴트'** 로만 처리합니다.

## 4. 저장매체·기기
- 개인 USB 사용은 금지되며, IT팀이 지급한 보안 USB만 사용할 수 있습니다.
- 업무용 노트북을 외부로 가져갈 때는 사전에 반출 신청을 합니다.
- 노트북·보안 USB 분실 시 **24시간 이내** IT헬프데스크에 신고합니다.

## 5. 계정·비밀번호
- 비밀번호는 10자 이상, 영문·숫자·특수문자 조합이어야 합니다.
- 90일마다 변경해야 합니다.
- 5회 연속 틀리면 계정이 잠기며, IT헬프데스크(내선 1234)로 해제를 요청합니다.

## 6. 클린데스크
- 퇴근 시 책상 위에 서류를 두지 않고, 대외비 문서는 잠금 서랍에 보관합니다.
"""

print(leave_policy[:150], "...")
print("\n✅ 규정 3개 준비 완료")

### 셀 7 — 규정을 보고 답하는 '현경봇' (= RAG)

프롬프트에 규정이 들어갈 자리 `{document}`를 만들고, **"규정에 없으면 모른다고 하라"** 고 지시합니다.

In [ ]:
rag_prompt = 

rag_chatbot = 

question = "현대경제연구소에서 못 쓴 연차는 다음 해로 며칠까지 넘길 수 있나요?"

answer = 
print(answer)

### 셀 8 — 나란히 비교 (문서 없이 vs 있이)

질문: **"배우자가 출산하면 휴가를 며칠 받을 수 있나요?"** (정답: 유급 10일, 90일 이내 사용)

In [ ]:
q = "배우자가 출산하면 휴가를 며칠 받을 수 있나요?"

print("───[ 문서 없이 ]───")
print()          # TODO: 셀 5의 일반 챗봇

print("\n───[ 문서 있이 (RAG) ]───")
print()   # TODO: RAG 챗봇 + 규정

### 셀 9 — 문서만 바꿔 끼우기 (출장·보안)

RAG 챗봇은 그대로 두고 `document`만 바꿉니다.

| 질문 | 넣을 문서 | 정답 |
|------|-----------|------|
| 부산 출장 숙박비는 1박에 얼마까지? | `travel_policy` | 10만원 (광역시) |
| ChatGPT에 미발표 성장률 전망치를 넣어도 되나요? | `security_policy` | 금지, '현경 어시스턴트' 사용 |

In [ ]:
q1 = "부산으로 출장 가는데 숙박비는 1박에 얼마까지 지원되나요?"
print("Q1:", rag_chatbot.invoke({"document": ____, "question": q1}))   # TODO

q2 = "ChatGPT에 미발표 경제성장률 전망치를 넣고 보고서 요약을 시켜도 되나요?"
print("\nQ2:", rag_chatbot.invoke({"document": ____, "question": q2}))  # TODO

### 셀 10 — 규정에 없는 질문 (환각 방지 확인)

규정 어디에도 **사내 헬스장** 이야기는 없습니다. 현경봇이 지어내지 않고 "모른다"고 하는지 확인합니다.

In [ ]:
q = "사내 헬스장은 몇 시까지 운영하나요?"

# TODO: 세 규정을 모두 이어 붙여 넣어보세요 (힌트: 문자열은 + 로 이어 붙일 수 있음)
all_docs = ____

print(rag_chatbot.invoke({"document": all_docs, "question": q}))

> ✅ "규정에서 확인되지 않습니다"라고 답하면 성공! 사내 챗봇은 **틀린 답보다 모른다는 답**이 안전합니다.

### 셀 11 — 문서를 다 넣으면 비용은? (다음 시간 예고)

세 규정을 통째로 넣을 때와 하나만 넣을 때의 **입력 토큰**을 비교합니다.

In [ ]:
q = "재택근무는 일주일에 몇 번까지 되나요?"

one = (rag_prompt | llm).invoke({"document": leave_policy, "question": q})
alldoc = (rag_prompt | llm).invoke({"document": all_docs, "question": q})

# TODO: 응답에서 입력 토큰 수를 꺼내는 키 이름은? (힌트: "input_tokens")
print("규정 1개 넣을 때:", one.usage_metadata[____])
print("규정 3개 넣을 때:", alldoc.usage_metadata[____])

> 🚪 질문과 관계없는 출장·보안 규정까지 넣으니 토큰이 몇 배로 늘었습니다.
> 실제 연구소 규정집은 수백 페이지입니다. 다음 시간에는 **질문에 필요한 부분만 골라 넣는 법**을 배웁니다.

# 5️⃣ 단계 — 미션: 현경봇 테스트하기

김연구 씨가 되어 직접 질문을 만들어 보세요.

| 미션 | 질문 예시 |
|------|-----------|
| ① 계산형 | "입사 5년차면 연차가 며칠이에요?" |
| ② 함정형 | "법인카드로 밤 11시 반에 야근 식사를 결제해도 되나요?" |
| ③ 여러 규정 | "해외 출장에 개인 USB를 가져가도 되나요?" |
| ④ 규정 밖 | "주차 지원은 어떻게 받나요?" |

In [ ]:
my_questions = [
    "____",   # ① 계산형
    "____",   # ② 함정형
    "____",   # ③ 여러 규정
    "____",   # ④ 규정 밖
]

for q in my_questions:
    print("🙋", q)
    print("🤖", rag_chatbot.invoke({"document": all_docs, "question": q}))
    print("-" * 60)

### 셀 13 — (도전) 현경봇 프롬프트 개선하기

답변 끝에 **근거 조항**과 **담당 부서**를 붙이도록 `system` 지시를 바꿔보세요.

> 예: "…입니다. (근거: 근무·휴가 규정 4. 미사용 연차 / 문의: 인사팀)"

In [ ]:
better_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "____"   # TODO: 역할 + 규정만 보고 답하기 + 근거 조항·담당 부서 표시 규칙
     "\n\n[규정]\n{document}"),
    ("human", "{question}"),
])
better_chatbot = better_prompt | llm | StrOutputParser()

print(better_chatbot.invoke({"document": all_docs, "question": "부모님이 돌아가시면 휴가와 경조금은?"}))